In [1]:
import os
import torch
import pandas as pd
import numpy as np
import pickle
from eval import season_performance_with_unlimited_transfers
import json
from model import FPLSequenceModel

In [2]:
base_path = os.getcwd()
base_path

'/Users/bragehs/Documents/FPL_forecast/predictor'

In [3]:
data_path = os.path.join(base_path, 'processed_data')
data_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [5]:
X_test_numeric = torch.load(data_path + '/X_test.pt', weights_only=True)
X_test_static = torch.load(data_path + '/X_static_test.pt', weights_only=True)
y_test = torch.load(data_path + '/y_test.pt', weights_only=True)
test_mapping = pd.read_csv(data_path + '/test_mapping.csv')

In [6]:
best_model_data = torch.load("best_model_with_mins.pth", map_location=torch.device('cpu'))

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_2816/2699782564.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model_data = torch.load("best_model_with_

In [7]:
print(best_model_data.keys())
for k, v in best_model_data['model_state_dict'].items():
    if 'embedding' in k:
        print(k, v.shape)

dict_keys(['model_state_dict', 'best_performance', 'hidden_dim', 'lstm_layers', 'dropout'])


In [9]:
print(best_model_data['hidden_dim'])
print(best_model_data['lstm_layers'])

128
2


In [12]:
model = FPLSequenceModel(
            numeric_seq_dim=X_test_numeric.shape[-1],
            static_dim=X_test_static.shape[-1],
            hidden_dim=best_model_data['hidden_dim'],
            lstm_layers=best_model_data["lstm_layers"],
            dropout=0.0,
            multitask=True
        )
model.load_state_dict(best_model_data['model_state_dict'])

<All keys matched successfully>

In [13]:
#print number of parameters in the model
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of parameters in the model: {num_params}")

Number of parameters in the model: 267414


In [14]:
model.eval()

FPLSequenceModel(
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (locked_dropout_in): LockedDropout()
  (lstm): LSTM(16, 128, num_layers=2, batch_first=True)
  (locked_dropout_out): LockedDropout()
  (attn_pool): AttentionPool(
    (proj): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=1, bias=True)
    )
  )
  (out_dropout): Dropout(p=0.0, inplace=False)
  (head_points): Sequential(
    (0): Linear(in_features=403, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=1, bias=True)
  )
  (head_minutes): Linear(in_features=403, out_features=1, bias=True)
)

In [15]:
test = pd.read_csv(data_path + '/test_data.csv')

In [16]:
test.columns

Index(['GW', 'last_1_assists', 'last_1_bonus', 'last_1_creativity',
       'last_1_clean_sheets', 'last_1_goals_conceded', 'last_1_goals_scored',
       'last_1_ict_index', 'last_1_influence', 'last_1_minutes',
       'last_1_threat', 'last_1_red_cards', 'last_1_yellow_cards',
       'last_1_team_score', 'last_1_opponent_score', 'last_all_assists',
       'last_all_bonus', 'last_all_creativity', 'last_all_clean_sheets',
       'last_all_goals_conceded', 'last_all_goals_scored',
       'last_all_ict_index', 'last_all_influence', 'last_all_minutes',
       'last_all_threat', 'last_all_red_cards', 'last_all_yellow_cards',
       'last_all_team_score', 'last_all_opponent_score', 'lagged_was_home',
       'lagged_fixture_difficulty', 'position_encoded_0.0',
       'position_encoded_1.0', 'position_encoded_2.0', 'position_encoded_3.0',
       'position_encoded_4.0', 'season_x', 'value', 'team_x', 'name',
       'element', 'minutes', 'was_home', 'total_points'],
      dtype='object')

In [18]:
points_predictions, mins_predictions = model(X_test_numeric, X_test_static)
points_predictions = points_predictions.detach().numpy()
mins_predictions = mins_predictions.detach().numpy()
print(points_predictions.shape)
print(mins_predictions.shape)
print(y_test.shape)

(27283, 1)
(27283, 1)
torch.Size([27283, 1])


In [31]:
def make_predicted_table(y_test, y_pred, mins_predictions):
    '''
    Create a DataFrame for LSTM model predictions.
    This needs to keep track of the Gameweek (GW) and player names.
    '''
    test_mapping = pd.read_csv(data_path + '/test_mapping.csv')
    predictions_df = test_mapping.copy()
    predictions_df['actual'] = y_test
    predictions_df['predicted'] = y_pred
    predictions_df['minutes_pred'] = mins_predictions
    predictions_df['predicted'] = predictions_df['predicted'].where(predictions_df['minutes'] > 0, 0)
    predictions_df.rename(columns={'prediction_gw': 'GW'}, inplace=True)


    # Update the predictions_df reference
    predictions_df = predictions_df.drop_duplicates(subset=['name', 'GW'], keep='last')
    
    return predictions_df


In [32]:
df = make_predicted_table(y_test, points_predictions, mins_predictions)

In [38]:
player = df[df['name'] == 'phil_foden']
player

,sequence_idx,element,season_x,name,GW,team_x,value,minutes,last_1_goals_scored,last_1_assists,padding_used,position_encoded,actual,predicted,minutes_pred
13186,13186,348.0,2024-25,phil_foden,1.0,NaN,95.0,45.0,0.00,0.00,4,NaN,1.0,0.267196,0.228821
13187,13187,348.0,2024-25,phil_foden,2.0,NaN,95.0,0.0,0.00,0.00,3,NaN,0.0,0.000000,41.879860
13188,13188,348.0,2024-25,phil_foden,3.0,NaN,94.0,0.0,0.00,0.00,2,NaN,0.0,0.000000,1.692575
13189,13189,348.0,2024-25,phil_foden,4.0,NaN,93.0,0.0,0.00,0.00,1,NaN,0.0,0.000000,-0.178102
13190,13190,348.0,2024-25,phil_foden,5.0,NaN,93.0,20.0,0.00,0.00,0,NaN,1.0,0.097436,-0.189664
13191,13191,348.0,2024-25,phil_foden,6.0,NaN,93.0,24.0,0.00,0.00,0,NaN,1.0,0.708353,11.289324
13192,13192,348.0,2024-25,phil_foden,7.0,NaN,92.0,77.0,0.00,0.00,0,NaN,2.0,1.163957,48.427479
13193,13193,348.0,2024-25,phil_foden,8.0,NaN,92.0,24.0,0.00,0.00,0,NaN,4.0,1.666824,75.753014
13194,13194,348.0,2024-25,phil_foden,9.0,NaN,92.0,90.0,0.00,0.25,0,NaN,3.0,1.608748,71.203056
13195,13195,348.0,2024-25,phil_foden,10.0,NaN,93.0,90.0,0.00,0.00,0,NaN,2.0,1.905745,78.611160


In [22]:
print(torch.mean(y_test))
print(torch.var(y_test))    

tensor(1.1469)
tensor(5.3394)


In [24]:
print(np.mean(points_predictions))
print(np.var(points_predictions))
print(np.max(points_predictions))

0.6529513
0.6448372
4.313949


In [25]:
print(np.mean(mins_predictions))
print(np.var(mins_predictions))
print(np.max(mins_predictions))

25.746609
1300.6908
89.24266


In [34]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

rmse = root_mean_squared_error(y_test, points_predictions)
print(f"RMSE: {rmse}")

mae = mean_absolute_error(y_test, points_predictions)
print(f"MAE: {mae}")


RMSE: 2.059161424636841
MAE: 0.8934563398361206


In [35]:
X_test_numeric.shape

torch.Size([27283, 5, 16])

In [39]:
scores, total_score = season_performance_with_unlimited_transfers(
    y_test=y_test,
    predictions=points_predictions,
)

Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
1.0 :  lukasz_fabianski
2.0 :  sepp_van_den_berg
3.0 :  tyler_dibling
4.0 :  daniel_jebbison
Bench players: ['lukasz_fabianski', 'sepp_van_den_berg', 'tyler_dibling', 'daniel_jebbison']
Bench cost: 170.0
Simulating season with unlimited transfers for 38 gameweeks
Available budget per gameweek: 830.0

--- Gameweek 1.0 ---
Players available for GW 1.0: 668
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/0fa97096062e45c8a50911ec7c12bb00-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/0fa97096062e45c8a50911ec7c12bb00-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 30 COLUMNS
At line 3687 RHS
At line 3713 BOUNDS
At line 

In [40]:
total_score.item()

2298.0

In [41]:
scores

,team,gw_score
0,"[ali_al_hamadi, christian_walton, daniel_caste...",23.0
1,"[bukayo_saka, danny_welbeck, joel_veltman, joe...",60.0
2,"[andrew_robertson, antonee_robinson, cristian_...",85.0
3,"[alisson_ramses_becker, antonee_robinson, brya...",44.0
4,"[bukayo_saka, chris_wood, david_raya_martin, d...",73.0
5,"[bryan_mbeumo, chris_wood, david_raya_martin, ...",40.0
6,"[cole_palmer, cristian_romero, danny_welbeck, ...",38.0
7,"[antonee_robinson, bryan_mbeumo, cole_palmer, ...",45.0
8,"[cole_palmer, diogo_dalot_teixeira, dwight_mcn...",56.0
9,"[bryan_mbeumo, chris_wood, cole_palmer, danny_...",59.0
